In [ ]:
nl_holidays = holidays.Netherlands(
    years=list(range(df.index.min().year, df.index.max().year + 2))
)

holiday_index = pd.DatetimeIndex(nl_holidays.keys())

feat = df.copy()

# Calendar features for current date
feat["day_of_week"] = feat.index.dayofweek
feat["week_of_year"] = feat.index.isocalendar().week.astype(int)
feat["month"] = feat.index.month
feat["year"] = feat.index.year
feat["day_of_month"] = feat.index.day
feat["is_weekend"] = (feat.index.dayofweek >= 5).astype(int)
feat["is_sunday"] = (feat.index.dayofweek == 6).astype(int)
feat["is_holiday"] = feat.index.normalize().isin(holiday_index).astype(int)
feat["end_of_month_flag"] = feat.index.is_month_end.astype(int)

# Cyclical encoding
feat["dow_sin"] = np.sin(2 * np.pi * feat["day_of_week"] / 7)
feat["dow_cos"] = np.cos(2 * np.pi * feat["day_of_week"] / 7)
feat["month_sin"] = np.sin(2 * np.pi * feat["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * feat["month"] / 12)

# Lag features
for lag in [1, 2, 3, 5, 7, 14, 21, 28]:
    feat[f"lag_{lag}"] = feat["orders"].shift(lag)

# Same weekday history
feat["same_weekday_avg_4"] = (
    feat["orders"].shift(7) +
    feat["orders"].shift(14) +
    feat["orders"].shift(21) +
    feat["orders"].shift(28)
) / 4

feat["same_weekday_last"] = feat["orders"].shift(7)

# Rolling features using past data only
feat["rolling_mean_3"] = feat["orders"].shift(1).rolling(3).mean()
feat["rolling_mean_7"] = feat["orders"].shift(1).rolling(7).mean()
feat["rolling_std_7"] = feat["orders"].shift(1).rolling(7).std()
feat["rolling_min_7"] = feat["orders"].shift(1).rolling(7).min()
feat["rolling_max_7"] = feat["orders"].shift(1).rolling(7).max()
feat["rolling_mean_14"] = feat["orders"].shift(1).rolling(14).mean()
feat["rolling_mean_28"] = feat["orders"].shift(1).rolling(28).mean()

# Trend and momentum features
feat["diff_1"] = feat["lag_1"] - feat["lag_2"]
feat["diff_7"] = feat["lag_1"] - feat["lag_7"]
feat["drop_7"] = feat["lag_7"] - feat["lag_14"]
feat["recent_vs_avg"] = feat["lag_1"] - feat["rolling_mean_7"]
feat["same_weekday_dev"] = feat["lag_7"] - feat["same_weekday_avg_4"]
feat["trend_7_vs_28"] = feat["rolling_mean_7"] - feat["rolling_mean_28"]
feat["ratio_7_to_28"] = feat["rolling_mean_7"] / (feat["rolling_mean_28"] + 1e-6)
feat["range_7"] = feat["rolling_max_7"] - feat["rolling_min_7"]

# Trend index
feat["trend"] = np.arange(len(feat))

# Remove rows with missing lag/rolling values
feat = feat.dropna().copy()

print("Feature table shape:", feat.shape)
feat.head()